In [1]:
#pip install pm4py
import os
os.getcwd()

'C:\\Users\\obami\\Documents\\Python_Pra\\Thesis_PPM_2024-25'

In [2]:
#import and preprocess data
import numpy as np
import pandas as pd
import pm4py
import joblib

#Encode Prefix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from keras.preprocessing.sequence import pad_sequences

In [3]:
df = pd.read_csv("dmd_df.csv")
#df = pd.read_csv("ptc_df.csv")
#df = pd.read_csv('helpdesk_df.csv')
df.head()

,timestamp,activity,case_id
0,2017-01-09 09:49:50+00:00,declaration submitted by employee,declaration 86791
1,2017-01-09 10:26:14+00:00,declaration submitted by employee,declaration 86795
2,2017-01-09 11:13:33+00:00,declaration submitted by employee,declaration 86800
3,2017-01-09 11:24:20+00:00,declaration submitted by employee,declaration 86731
4,2017-01-09 11:27:48+00:00,declaration final_approved by supervisor,declaration 86791


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56437 entries, 0 to 56436
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   timestamp  56437 non-null  object
 1   activity   56437 non-null  object
 2   case_id    56437 non-null  object
dtypes: object(3)
memory usage: 1.3+ MB


In [5]:
df["timestamp"] = pd.to_datetime(df["timestamp"]) # conversion from object to date type

In [6]:
#split data test
from split_train_test import split_train_test_temporal
train, test, fp_dict = split_train_test_temporal(df,0.2,"case_id","timestamp","preferred")

#split data validation
train, val, fp_dict = split_train_test_temporal(train,0.1,"case_id","timestamp","preferred")

In [8]:
from bos_eos import add_bos_eos_target
############## train data transformation ########################
train_prefix = add_bos_eos_target(train,"prefix")
train_target = add_bos_eos_target(train)

############## validation data transformation ###################
val_prefix = add_bos_eos_target(val,"prefix")
val_target = add_bos_eos_target(val)

############## test data transformation ########################
test_prefix = add_bos_eos_target(test,"prefix")
test_target = add_bos_eos_target(test)

In [9]:
train_prefix_copy = train_prefix.copy()
val_prefix_copy = val_prefix.copy()
test_prefix_copy = test_prefix.copy()

#get complete cases
cases = df["activity"].unique()

In [10]:
from prefix_trace import prefix_trace
############## train data transformation ########################
train_prefix_trace = prefix_trace(train_prefix)

############## validation data transformation ###################
val_prefix_trace = prefix_trace(val_prefix)

############## test data transformation ########################
test_prefix_trace = prefix_trace(test_prefix)

In [11]:
train_prefix_trace.head()

,case_id,prefix
0,declaration 100000,[BOS]
1,declaration 100000,"[BOS, declaration submitted by employee]"
2,declaration 100000,"[BOS, declaration submitted by employee, decla..."
3,declaration 100000,"[BOS, declaration submitted by employee, decla..."
4,declaration 100000,"[BOS, declaration submitted by employee, decla..."


In [12]:
from encoding import find_max_len, si_encoding

#all possible cases. I assume in a business all activities are already known and defined
cases = np.append(cases,"BOS")
cases = np.append(cases,"zos")

#find the maximum length of the longest case for padding
max_len = find_max_len(train_prefix_trace["prefix"],val_prefix_trace["prefix"],test_prefix_trace["prefix"])

from encoding import si_encoding
############## train data transformation ########################
train_prefix_trace_encoded, label_encoder = si_encoding(train_prefix_trace,cases,max_len)
train_target_encoded, a = si_encoding(train_target,cases,option = "target")

In [13]:
############## validation data transformation ###################
val_prefix_trace_encoded, a = si_encoding(val_prefix_trace,cases,max_len)
val_target_encoded, a = si_encoding(val_target,cases,option="target")

In [14]:
############## test data transformation ########################
test_prefix_trace_encoded, a = si_encoding(test_prefix_trace,cases,max_len)
test_target_encoded, a = si_encoding(test_target,cases,option="target")

In [15]:
train_target_encoded

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 0., 1.]])

In [16]:
from dfg_probabilities import dfg_df

############ train #################
#get probability
probability = dfg_df(train_prefix,cases)

#encode labels for probability index
probability.index = label_encoder.transform(probability.index)
probability.columns = label_encoder.transform(probability.columns)

#reset index
probability.reset_index(inplace=True)
probability.rename(columns = {"index":"activity"},inplace=True)

#encode drop extra columns and encode activity
train_prefix["activity"] = label_encoder.transform(train_prefix["activity"])

#merge to get new dataframe
train_dfg_probability = pd.merge(train_prefix,probability,how="left",on="activity")
train_dfg_probability = train_dfg_probability.drop(columns = ["timestamp","case_id"])

In [17]:
############ validation ################################## 
#encode drop extra columns and encode activity
val_prefix["activity"] = label_encoder.transform(val_prefix["activity"])

#merge to get new dataframe. probability is same as train
val_dfg_probability = pd.merge(val_prefix,probability,how="left",on="activity")
val_dfg_probability = val_dfg_probability.drop(columns = ["timestamp","case_id"])

In [18]:
############ test ################################## 
#probability is combination of train and validation
train_val_prefix = pd.concat([train_prefix_copy,val_prefix])
probability = dfg_df(train_val_prefix,cases)

#encode drop extra columns and encode activity
test_prefix["activity"] = label_encoder.transform(test_prefix["activity"])

#encode labels for probability index
probability.index = label_encoder.transform(probability.index)
probability.columns = label_encoder.transform(probability.columns)

#reset index
probability.reset_index(inplace=True)
probability.rename(columns = {"index":"activity"},inplace=True)

#merge to get new dataframe
test_dfg_probability = pd.merge(test_prefix,probability,how="left",on="activity")
test_dfg_probability = test_dfg_probability.drop(columns = ["timestamp","case_id"])

In [ ]:
########################################### Help Desk ###########################################################################

##prefix data
np.save("helpdesk_train_prefix.npy",train_prefix_trace_encoded)
np.save("helpdesk_val_prefix.npy",val_prefix_trace_encoded)
np.save("helpdesk_test_prefix.npy",test_prefix_trace_encoded)

##probability data
train_dfg_probability.to_csv("helpdesk_train_dfg_probability.csv",index=False)
val_dfg_probability.to_csv("helpdesk_val_dfg_probability.csv",index=False)
test_dfg_probability.to_csv("helpdesk_test_dfg_probability.csv",index=False)

#target
np.save("helpdesk_train_target.npy",train_target_encoded)
np.save("helpdesk_val_target.npy",val_target_encoded)
np.save("helpdesk_test_target.npy",test_target_encoded)

#original data
train_target.to_csv("helpdesk_train_target_org.csv",index=False)
test_target.to_csv("helpdesk_test_target_org.csv",index=False)

In [ ]:
######################################### BPI Data ###############################################################################

##prefix data
np.save("ptc_train_prefix.npy",train_prefix_trace_encoded)
np.save("ptc_val_prefix.npy",val_prefix_trace_encoded)
np.save("ptc_test_prefix.npy",test_prefix_trace_encoded)

##probability data
train_dfg_probability.to_csv("ptc_train_dfg_probability.csv",index=False)
val_dfg_probability.to_csv("ptc_val_dfg_probability.csv",index=False)
test_dfg_probability.to_csv("ptc_test_dfg_probability.csv",index=False)

#target
np.save("ptc_train_target.npy",train_target_encoded)
np.save("ptc_val_target.npy",val_target_encoded)
np.save("ptc_test_target.npy",test_target_encoded)

#original data
train_target.to_csv("ptc_train_target_org.csv",index=False)
test_target.to_csv("ptc_test_target_org.csv",index=False)

In [ ]:
######################################### RMP Data ###############################################################################

##prefix data
np.save("dmd_train_prefix.npy",train_prefix_trace_encoded)
np.save("dmd_val_prefix.npy",val_prefix_trace_encoded)
np.save("dmd_test_prefix.npy",test_prefix_trace_encoded)

##probability data
train_dfg_probability.to_csv("dmd_train_dfg_probability.csv",index=False)
val_dfg_probability.to_csv("dmd_val_dfg_probability.csv",index=False)
test_dfg_probability.to_csv("dmd_test_dfg_probability.csv",index=False)

#target
np.save("dmd_train_target.npy",train_target_encoded)
np.save("dmd_val_target.npy",val_target_encoded)
np.save("dmd_test_target.npy",test_target_encoded)

#original data
train_target.to_csv("dmd_train_target_org.csv",index=False)
test_target.to_csv("dmd_test_target_org.csv",index=False)

In [ ]:
# ptc 
#joblib.dump(label_encoder, 'ptc_label_encoder.joblib'

In [ ]:
# Helpdesk 
joblib.dump(label_encoder, 'helpdesk_label_encoder.joblib')

In [19]:
# dmd 
joblib.dump(label_encoder, 'dmd_label_encoder.joblib')

['dmd_label_encoder.joblib']

In [32]:
)
# Helpdesk 
joblib.dump(label_encoder, 'helpdesk_label_encoder.joblib')

['helpdesk_label_encoder.joblib']

In [20]:
train_prefix_copy.head()

,timestamp,activity,case_id
0,2018-01-30 09:20:06+00:00,BOS,declaration 100000
1,2018-01-30 09:20:07+00:00,declaration submitted by employee,declaration 100000
2,2018-02-07 09:58:46+00:00,declaration approved by administration,declaration 100000
3,2018-02-08 10:59:05+00:00,declaration final_approved by supervisor,declaration 100000
4,2018-02-09 12:42:49+00:00,request payment,declaration 100000


In [21]:
train_prefix_copy.head()

,timestamp,activity,case_id
0,2018-01-30 09:20:06+00:00,BOS,declaration 100000
1,2018-01-30 09:20:07+00:00,declaration submitted by employee,declaration 100000
2,2018-02-07 09:58:46+00:00,declaration approved by administration,declaration 100000
3,2018-02-08 10:59:05+00:00,declaration final_approved by supervisor,declaration 100000
4,2018-02-09 12:42:49+00:00,request payment,declaration 100000


In [22]:
cases

array(['declaration submitted by employee',
       'declaration final_approved by supervisor', 'request payment',
       'declaration rejected by missing', 'declaration saved by employee',
       'declaration approved by pre_approver', 'payment handled',
       'declaration rejected by supervisor',
       'declaration rejected by pre_approver',
       'declaration rejected by employee',
       'declaration for_approval by supervisor',
       'declaration for_approval by pre_approver',
       'declaration approved by administration',
       'declaration approved by budget owner',
       'declaration rejected by administration',
       'declaration rejected by budget owner',
       'declaration for_approval by administration', 'BOS', 'zos'],
      dtype=object)

In [23]:
from dfg_probabilities import dfg_df

############ train #################
#get probability

probability = dfg_df(train_prefix_copy,cases)

In [24]:
probability.head()

,declaration approved by administration,declaration rejected by administration,declaration approved by pre_approver,declaration rejected by supervisor,declaration rejected by pre_approver,declaration rejected by employee,declaration for_approval by administration,declaration final_approved by supervisor,declaration for_approval by supervisor,declaration for_approval by pre_approver,request payment,declaration rejected by missing,payment handled,declaration submitted by employee,declaration approved by budget owner,declaration rejected by budget owner,declaration saved by employee,zos
declaration submitted by employee,0.628684,0.069614,0.087017,0.007622,0.010925,0.000635,0.000127,0.195122,0.000127,0.000127,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
declaration approved by pre_approver,0.000000,0.000000,0.000000,0.016058,0.000000,0.000000,0.000000,0.983942,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
declaration approved by administration,0.000000,0.000000,0.000000,0.022429,0.000000,0.000000,0.000000,0.627804,0.000000,0.000000,0.0,0.0,0.0,0.0,0.341281,0.008487,0.0,0.0
declaration approved by budget owner,0.000000,0.000000,0.000000,0.008289,0.000000,0.000000,0.000000,0.991711,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0
payment handled,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0


In [25]:
#predict prefix, evaluate on target

In [38]:
next_activity = probability.idxmax(axis=1)
train_prefix_copy["next_activity"] = train_prefix_copy["activity"].map(next_activity)

In [40]:
train_prefix_copy["next_activity"].unique()

array(['declaration submitted by employee',
       'declaration approved by administration',
       'declaration final_approved by supervisor', 'request payment',
       'payment handled', 'declaration rejected by employee',
       'declaration rejected by missing'], dtype=object)

In [28]:
#result

In [41]:
train_prefix_copy.head()

,timestamp,activity,case_id,next_activity
0,2018-01-30 09:20:06+00:00,BOS,declaration 100000,declaration submitted by employee
1,2018-01-30 09:20:07+00:00,declaration submitted by employee,declaration 100000,declaration approved by administration
2,2018-02-07 09:58:46+00:00,declaration approved by administration,declaration 100000,declaration final_approved by supervisor
3,2018-02-08 10:59:05+00:00,declaration final_approved by supervisor,declaration 100000,request payment
4,2018-02-09 12:42:49+00:00,request payment,declaration 100000,payment handled


In [42]:
train_prefix_copy.fillna("none",inplace=True)

In [44]:
#result in report
from sklearn.metrics import multilabel_confusion_matrix, classification_report
report = classification_report(train_target["activity"],train_prefix_copy["next_activity"],zero_division=0)
print(report)

                                            precision    recall  f1-score   support

    declaration approved by administration       0.63      1.00      0.77      4949
      declaration approved by budget owner       0.00      0.00      0.00      1689
      declaration approved by pre_approver       0.00      0.00      0.00       685
  declaration final_approved by supervisor       0.75      0.78      0.76      6992
declaration for_approval by administration       0.00      0.00      0.00         1
  declaration for_approval by pre_approver       0.00      0.00      0.00         1
    declaration for_approval by supervisor       0.00      0.00      0.00         1
    declaration rejected by administration       0.00      0.00      0.00       548
      declaration rejected by budget owner       0.00      0.00      0.00        42
          declaration rejected by employee       0.11      0.99      0.20       865
           declaration rejected by missing       1.00      0.02      0.04  